[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_18/MFM_ch18_spillover_uncertainty/MFM_ch18_spillover_uncertainty.ipynb)

# MFM_ch18_spillover_uncertainty

Estimation error in the Diebold-Yilmaz network of 13 banks: residual block-bootstrap intervals for total and net connectedness, the adaptive elastic-net VAR and 150-day elastic-net windows of Demirer, Diebold, Liu and Yilmaz (2018) against OLS, and pairwise Granger-causality networks with Holm and Benjamini-Hochberg control.

*Modelling Financial Markets (MFM), Chapter 18: Systemic Risk and Networks. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_18'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from scipy.stats import spearmanr
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from sklearn.linear_model import QuantileRegressor, ElasticNetCV
from scipy import optimize
try:
    from arch import arch_model
except ImportError:                      # Colab: pachetul arch nu este preinstalat
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])
    from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
REGION_COL = {'US': MainBlue, 'EU': IDAred, 'RO': Forest}
REGION_LAB = {'US': 'US banks', 'EU': 'European banks', 'RO': 'Romanian banks (BVB)'}
EVENTS = [('2011-08-08', 'Euro crisis'), ('2016-06-24', 'Brexit vote'), ('2018-12-19', 'RO bank tax'),
          ('2020-03-16', 'COVID-19'), ('2023-03-13', 'SVB'), ('2025-04-07', 'Tariffs')]

HERE = '.'
SEED = 42
B_BOOT = 999
RESULTS = {}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return [jsonable(v) for v in x.tolist()]
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, pd.Timestamp):
        return str(x.date())
    return x


def mark_events(ax, ymax=None):
    """Linii verticale de referinta pentru evenimente, cu eticheta neagra."""
    for d, lab in EVENTS:
        t = pd.Timestamp(d)
        if ax.get_xlim()[0] <= mdates.date2num(t) <= ax.get_xlim()[1]:
            ax.axvline(t, color=Gray, lw=0.5, ls=':')
            ax.text(t, ax.get_ylim()[1] if ymax is None else ymax, lab, rotation=90, fontsize=6.5, color='black',
                    ha='right', va='top')

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
START = '2010-01-01'
END = '2026-09-18'

# banca -> (simbol, nume, regiune)
BANKS = {
    'JPM':  ('JPM.US',    'JPMorgan Chase',       'US'),
    'BAC':  ('BAC.US',    'Bank of America',      'US'),
    'C':    ('C.US',      'Citigroup',            'US'),
    'GS':   ('GS.US',     'Goldman Sachs',        'US'),
    'MS':   ('MS.US',     'Morgan Stanley',       'US'),
    'WFC':  ('WFC.US',    'Wells Fargo',          'US'),
    'DBK':  ('DBK.XETRA', 'Deutsche Bank',        'EU'),
    'BNP':  ('BNP.PA',    'BNP Paribas',          'EU'),
    'SAN':  ('SAN.MC',    'Santander',            'EU'),
    'INGA': ('INGA.AS',   'ING',                  'EU'),
    'HSBA': ('HSBA.LSE',  'HSBC',                 'EU'),
    'TLV':  ('TLV.RO',    'Banca Transilvania',   'RO'),
    'BRD':  ('BRD.RO',    'BRD-Groupe SG',        'RO'),
}
US = [k for k, v in BANKS.items() if v[2] == 'US']
EU = [k for k, v in BANKS.items() if v[2] == 'EU']
RO = [k for k, v in BANKS.items() if v[2] == 'RO']
ALL = US + EU + RO
NAMES = {k: v[1] for k, v in BANKS.items()}
INDICES = {'SPX': ('GSPC.INDX', 'S&P 500'), 'SX5E': ('STOXX50E.INDX', 'Euro Stoxx 50'),
           'BET': ('BET', 'BET'), 'VIX': ('VIX.INDX', 'VIX')}
REGION_INDEX = {'US': 'SPX', 'EU': 'SX5E', 'RO': 'BET'}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def bank_frame(key, start=START, end=END):
    """OHLCV al unei banci, cu pretul ajustat si corectiile de mai sus."""
    symbol, _, region = BANKS[key]
    d = read_market(symbol).loc[start:end].copy()
    d = d[(d.index.dayofweek < 5) & (d['adjusted_close'] > 0)]
    if region == 'RO':
        d = d[d['volume'] > 0]                                   # zile fara tranzactii
    if key == 'TLV' and pd.Timestamp('2016-05-30') in d.index:
        # ajustarea pentru actiunile gratuite: factorul de dupa data ex se aplica si pe 30 mai 2016
        f = d.loc['2016-05-31', 'adjusted_close'] / d.loc['2016-05-31', 'close']
        d.loc['2016-05-30', 'adjusted_close'] = d.loc['2016-05-30', 'close'] * f
    return d


def bank_price(key, start=START, end=END):
    return bank_frame(key, start, end)['adjusted_close'].rename(key)


def index_close(key, start=START, end=END):
    s = read_market(INDICES[key][0])['close'].loc[start:end]
    s = s[(s.index.dayofweek < 5) & (s > 0)].dropna()
    if key != 'VIX':
        s = s[s.diff() != 0]                                     # sarbatori completate cu pretul anterior
    return s.rename(key)


def prices(keys, start=START, end=END):
    """Preturi zilnice pentru banci si/sau indici, doar in zilele comune."""
    cols = [bank_price(k, start, end) if k in BANKS else index_close(k, start, end) for k in keys]
    return pd.concat(cols, axis=1, join='inner').dropna()


def joint_returns(keys, start=START, end=END):
    """Randamente log zilnice in %: intai join pe PRETURI in zilele comune, apoi randamente."""
    return 100 * np.log(prices(keys, start, end)).diff().dropna()


def bank_range_vol(keys, start=START, end=END):
    """Volatilitatea zilnica Parkinson (in %, anualizata), in zilele comune tuturor bancilor."""
    out = []
    for k in keys:
        d = bank_frame(k, start, end)
        rng = np.log(d['high'] / d['low']).where(d['high'] > d['low'])
        s2 = rng ** 2 / (4 * np.log(2))
        floor = s2.where(s2 > 0).cummin().ffill()                  # cea mai mica valoare pozitiva observata PANA la ziua t
        s2 = s2.fillna(floor)                                     # zile cu maxim = minim: fara informatie din viitor
        out.append((100 * np.sqrt(252 * s2)).rename(k))
    return pd.concat(out, axis=1, join='inner').dropna()


def system_return(R, members):
    """Randamentul sistemului: portofoliul cu ponderi egale al bancilor (randamente log in %, aproximare zilnica)."""
    return R[members].mean(axis=1)


# randamente log zilnice in % (zile comune) si volatilitatea Parkinson (log)
R = joint_returns(ALL + ['SPX', 'SX5E', 'BET', 'VIX'])
V = np.log(bank_range_vol(ALL))
print(R.index[0].date(), '->', R.index[-1].date(), len(R), 'common days')


def system_ex(k, members):
    """Randamentul sistemului regional fara banca k (portofoliu cu ponderi egale)."""
    return R[[j for j in members if j != k]].mean(axis=1)


def region_of(k):
    return BANKS[k][2]

## Systemic risk measures

In [ ]:
def mes(r_i, r_m, alpha=0.05):
    """MES (%): pierderea medie a bancii in zilele in care piata este in coada de probabilitate alpha."""
    q = np.quantile(r_m, alpha)
    return -np.mean(np.asarray(r_i)[np.asarray(r_m) <= q])


def mes_threshold(r_i, r_m, c=-2.0):
    """MES (%) in zilele in care piata scade cu mai mult de |c|% (definitia folosita pentru LRMES)."""
    return -np.mean(np.asarray(r_i)[np.asarray(r_m) < c])


def lrmes(mes2_pct):
    """Pierderea pe termen lung intr-o criza (piata -40% in sase luni): LRMES = 1 - exp(-18 MES), MES in fractii."""
    return 1 - np.exp(-18 * mes2_pct / 100)


def srisk_ratio(lrm, leverage, k=0.08):
    """SRISK / capitalul de piata W: k(L - 1) - (1 - k)(1 - LRMES), cu L = (D + W)/W."""
    return k * (leverage - 1) - (1 - k) * (1 - lrm)


def breakeven_leverage(lrm, k=0.08):
    """Levierul L* peste care SRISK > 0: L* = 1 + (1 - k)(1 - LRMES)/k."""
    return 1 + (1 - k) * (1 - lrm) / k


def qreg(y, X, q):
    """Regresie cuantila (Koenker-Bassett); intoarce coeficientii [termen liber, pante]."""
    return QuantReg(np.asarray(y), sm.add_constant(np.asarray(X), has_constant='add')).fit(q=q, max_iter=5000).params


def covar_static(x_sys, x_i, alpha=0.01):
    """CoVaR si Delta-CoVaR necondiționate (%, pozitive = pierderi).

    Regresia cuantila la nivelul alpha: X_sys = a + b X_i;
    CoVaR_alpha = -(a + b q_alpha(X_i)); Delta-CoVaR = b (q_50(X_i) - q_alpha(X_i))."""
    x_sys, x_i = np.asarray(x_sys), np.asarray(x_i)
    a, b = qreg(x_sys, x_i, alpha)
    qa, qm = np.quantile(x_i, alpha), np.quantile(x_i, 0.5)
    return {'a': a, 'b': b, 'var_i': -qa, 'covar': -(a + b * qa), 'covar_med': -(a + b * qm),
            'dcovar': b * (qm - qa), 'var_sys': -np.quantile(x_sys, alpha)}


def block_bootstrap_idx(n, block=20, rng=None):
    """Indici pentru bootstrap pe blocuri mobile (Künsch), blocuri de lungime fixa."""
    rng = rng or np.random.default_rng(0)
    starts = rng.integers(0, n - block + 1, size=int(np.ceil(n / block)))
    return (starts[:, None] + np.arange(block)[None, :]).ravel()[:n]


def covar_boot(x_sys, x_i, alpha=0.01, B=200, block=20, seed=0):
    """Interval percentil de 95% pentru Delta-CoVaR prin bootstrap pe blocuri mobile."""
    x_sys, x_i = np.asarray(x_sys), np.asarray(x_i)
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(B):
        idx = block_bootstrap_idx(len(x_i), block, rng)
        d.append(covar_static(x_sys[idx], x_i[idx], alpha)['dcovar'])
    return np.percentile(d, [2.5, 97.5])


def covar_dynamic(x_sys, x_i, M, alpha=0.01):
    """Delta-CoVaR variabil in timp cu variabile de stare M_{t-1} (Adrian & Brunnermeier, 2016).

    X_i,t = a + c'M_{t-1} la nivelurile alpha si 50%  ->  VaR_i,t;
    X_sys,t = a + b X_i,t + c'M_{t-1} la nivelul alpha  ->  Delta-CoVaR_t = b (q50_i,t - qalpha_i,t)."""
    ci_a = qreg(x_i, M, alpha)
    ci_m = qreg(x_i, M, 0.5)
    cs = qreg(x_sys, np.column_stack([x_i, M]), alpha)
    Mc = sm.add_constant(np.asarray(M), has_constant='add')
    qa, qm = Mc @ ci_a, Mc @ ci_m
    return pd.DataFrame({'var_i': -qa, 'dcovar': cs[1] * (qm - qa)}, index=getattr(x_i, 'index', None)), cs[1]


def var_fit(Y, p):
    """VAR(p) estimat prin OLS; intoarce coeficientii A_1..A_p (k x k) si matricea de covarianta a reziduurilor."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([np.ones((T - p, 1))] + [Y[p - l - 1:T - l - 1] for l in range(p)])
    Yt = Y[p:]
    B = np.linalg.lstsq(X, Yt, rcond=None)[0]
    E = Yt - X @ B
    S = E.T @ E / (len(Yt) - X.shape[1])
    A = [B[1 + l * k:1 + (l + 1) * k].T for l in range(p)]
    return A, S


def var_bic(Y, pmax=5):
    """Ordinul VAR ales prin criteriul BIC (Schwarz), cu matricea de covarianta a reziduurilor de verosimilitate maxima."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    best = None
    n = T - pmax
    for p in range(1, pmax + 1):
        A, S = var_fit(Y[pmax - p:], p)
        S_ml = S * (n - 1 - p * k) / n                        # covarianta de verosimilitate maxima E'E / n
        bic = np.log(np.linalg.det(S_ml)) + np.log(n) * p * k * k / n
        if best is None or bic < best[1]:
            best = (p, bic)
    return best[0]


def ma_coefs(A, H):
    """Coeficientii reprezentarii medie mobila Phi_0..Phi_{H-1}."""
    k = A[0].shape[0]
    Phi = [np.eye(k)]
    for h in range(1, H):
        Phi.append(sum(A[l] @ Phi[h - l - 1] for l in range(min(h, len(A)))))
    return Phi


def gfevd(A, S, H=10):
    """Descompunerea generalizata a dispersiei erorii de prognoza (Pesaran-Shin), normalizata pe linii."""
    Phi = ma_coefs(A, H)
    k = S.shape[0]
    num = np.zeros((k, k))
    den = np.zeros(k)
    for P in Phi:
        PS = P @ S
        num += PS ** 2
        den += np.diag(P @ S @ P.T)
    theta = num / np.diag(S)[None, :] / den[:, None]
    return theta / theta.sum(axis=1, keepdims=True)


def spillover_table(theta, names):
    """Tabelul de conectivitate: contributii (%), FROM, TO, NET si indicele total."""
    k = theta.shape[0]
    D = 100 * theta
    off = D - np.diag(np.diag(D))
    frm, to = off.sum(axis=1), off.sum(axis=0)
    tab = pd.DataFrame(D, index=names, columns=names)
    return {'table': tab, 'from': pd.Series(frm, index=names), 'to': pd.Series(to, index=names),
            'net': pd.Series(to - frm, index=names), 'total': off.sum() / k,
            'pairwise_net': pd.DataFrame(off.T - off, index=names, columns=names)}


def rolling_total(Y, window=250, step=5, p=2, H=10):
    """Indicele total de conectivitate pe ferestre mobile."""
    Y = pd.DataFrame(Y)
    out, frm_to = {}, {}
    for end in range(window, len(Y) + 1, step):
        w = Y.iloc[end - window:end]
        A, S = var_fit(w.values, p)
        st = spillover_table(gfevd(A, S, H), list(Y.columns))
        out[Y.index[end - 1]] = st['total']
        frm_to[Y.index[end - 1]] = st['net']
    return pd.Series(out), pd.DataFrame(frm_to).T


def check_loss(u, tau):
    return u * (tau - (u < 0))


def lasso_qr_gacv(y, X, tau=0.05, lambdas=None):
    """Regresie cuantila penalizata L1; lambda ales prin GACV = sum rho_tau(rez) / (n - df)."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    n = len(y)
    lambdas = np.logspace(-4, -0.5, 15) if lambdas is None else lambdas
    rows = []
    for lam in lambdas:
        m = QuantileRegressor(quantile=tau, alpha=lam, solver='highs').fit(X, y)
        res = y - m.predict(X)
        df = int(np.sum(np.abs(m.coef_) > 1e-8))
        g = check_loss(res, tau).sum() / max(n - df, 1)
        rows.append((lam, g, df, m.coef_.copy()))
    i = int(np.argmin([r[1] for r in rows]))
    return {'lambda': rows[i][0], 'gacv': [r[1] for r in rows], 'df': [r[2] for r in rows],
            'lambdas': list(lambdas), 'coef': rows[i][3], 'df_sel': rows[i][2]}

## Helper functions

In [ ]:
B = 999

## Analysis

In [ ]:
def holm(p):
    """Corectia Holm (valori p ajustate) pentru o familie de teste."""
    p = np.asarray(p, float)
    o = np.argsort(p)
    m = len(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[o])
    out = np.empty(m)
    out[o] = np.minimum(adj, 1)
    return out


def bh(p, q=0.05):
    """Benjamini-Hochberg: masca respingerilor la rata de descoperiri false q."""
    p = np.asarray(p, float)
    m = len(p)
    o = np.argsort(p)
    ok = p[o] <= q * np.arange(1, m + 1) / m
    k = np.max(np.nonzero(ok)[0]) + 1 if ok.any() else 0
    rej = np.zeros(m, bool)
    rej[o[:k]] = True
    return rej


def var_simulate(A, c, E, Y0, idx):
    p = len(A)
    T = len(idx)
    k = A[0].shape[0]
    Y = np.zeros((T + p, k))
    Y[:p] = Y0
    for t in range(p, T + p):
        Y[t] = c + sum(A[l] @ Y[t - l - 1] for l in range(p)) + E[idx[t - p]]
    return Y


def var_fit_c(Y, p):
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([np.ones((T - p, 1))] + [Y[p - l - 1:T - l - 1] for l in range(p)])
    Bc = np.linalg.lstsq(X, Y[p:], rcond=None)[0]
    E = Y[p:] - X @ Bc
    return [Bc[1 + l * k:1 + (l + 1) * k].T for l in range(p)], Bc[0], E


def spill_bootstrap(Y, p, H=10, B=B, block=20, seed=SEED):
    """Interval percentil pentru conectivitatea totala si neta (bootstrap pe blocuri de reziduuri, VAR reestimat)."""
    Y = np.asarray(Y, float)
    A, c, E = var_fit_c(Y, p)
    rng = np.random.default_rng(seed)
    tot, net = [], []
    for _ in range(B):
        idx = block_bootstrap_idx(len(E), block, rng)
        Yb = var_simulate(A, c, E, Y[:p], idx)
        Ab, Sb = var_fit(Yb, p)
        st = spillover_table(gfevd(Ab, Sb, H), list(range(Y.shape[1])))
        tot.append(st['total'])
        net.append(st['net'].values)
    return np.array(tot), np.array(net)


def fig_spill_ci(point_net, net_draws):
    lo, hi = np.percentile(net_draws, [2.5, 97.5], axis=0)
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    xs = np.arange(len(ALL))
    ax.bar(xs, point_net, color=[REGION_COL[region_of(k)] for k in ALL], width=0.6)
    ax.errorbar(xs, point_net, yerr=[point_net - lo, hi - point_net], fmt='none', ecolor='black', capsize=2, lw=0.8)
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xticks(xs)
    ax.set_xticklabels([NAMES[k] for k in ALL], rotation=35, ha='right', fontsize=7.5)
    ax.set_ylabel('Net spillover (to - from, %)')
    h = [plt.Rectangle((0, 0), 1, 1, color=REGION_COL[r]) for r in ['US', 'EU', 'RO']] + \
        [plt.Line2D([], [], color='black', marker='|', ls='', ms=8)]
    ax.legend(h, ['US banks', 'European banks', 'Romanian banks', '95% residual block-bootstrap interval'],
              loc='upper center', bbox_to_anchor=(0.5, -0.42), ncol=2, frameon=False)
    save_fig('ch18_spill_ci')
    return lo, hi


def aenet_eq(X, y, w, lambdas, folds=10, seed=SEED):
    """min sum (y - Xb)^2 + lam sum w_i (|b_i|/2 + b_i^2/2)  (ecuatia (9)); lambda prin validare incrucisata pe 10 grupe."""
    def fit(X, y, lam):
        Xc, yc = X - X.mean(0), y - y.mean()
        z = (Xc ** 2).sum(0)
        b = np.zeros(X.shape[1])
        r = yc.copy()
        for _ in range(200):
            b_old = b.copy()
            for j in range(len(b)):
                r += Xc[:, j] * b[j]
                rho = Xc[:, j] @ r
                b[j] = np.sign(rho) * max(abs(rho) - lam * w[j] / 4, 0) / (z[j] + lam * w[j] / 2)
                r -= Xc[:, j] * b[j]
            if np.max(np.abs(b - b_old)) < 1e-7:
                break
        return y.mean() - X.mean(0) @ b, b
    rng = np.random.default_rng(seed)
    fid = rng.permutation(len(y)) % folds
    cv = []
    for lam in lambdas:
        err = 0
        for f in range(folds):
            tr, te = fid != f, fid == f
            c0, b = fit(X[tr], y[tr], lam)
            err += ((y[te] - c0 - X[te] @ b) ** 2).sum()
        cv.append(err)
    lam = lambdas[int(np.argmin(cv))]
    return fit(X, y, lam), lam


def var_aenet(Y, p, lambdas=None):
    """VAR(p) cu elastic net adaptiv ecuatie cu ecuatie; ponderi w = 1/|b_OLS|."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([Y[p - l - 1:T - l - 1] for l in range(p)])
    Yt = Y[p:]
    lambdas = np.logspace(0, 5, 26) if lambdas is None else lambdas
    Bm = np.zeros((k * p, k))
    c = np.zeros(k)
    nz, lam_sel = 0, []
    for i in range(k):
        bols = np.linalg.lstsq(np.column_stack([np.ones(len(X)), X]), Yt[:, i], rcond=None)[0][1:]
        (c[i], Bm[:, i]), lam = aenet_eq(X, Yt[:, i], 1 / np.abs(bols), lambdas)
        nz += int((np.abs(Bm[:, i]) > 1e-10).sum())
        lam_sel.append(lam)
    E = Yt - c - X @ Bm
    S = E.T @ E / (len(Yt) - 1)
    A = [Bm[l * k:(l + 1) * k].T for l in range(p)]
    return A, S, nz / (k * k * p)


def var_enet_window(Y, p):
    """Elastic net (w_i = 1) pentru ferestrele mobile; penalizarea L1:L2 = 1:1 din ecuatia (9) <=> l1_ratio = 1/3."""
    T, k = Y.shape
    X = np.hstack([Y[p - l - 1:T - l - 1] for l in range(p)])
    Yt = Y[p:]
    Bm = np.zeros((k * p, k))
    c = np.zeros(k)
    for i in range(k):
        m = ElasticNetCV(l1_ratio=1 / 3, cv=10, n_alphas=30, max_iter=5000).fit(X, Yt[:, i])
        Bm[:, i], c[i] = m.coef_, m.intercept_
    E = Yt - c - X @ Bm
    S = E.T @ E / (len(Yt) - 1)
    return [Bm[l * k:(l + 1) * k].T for l in range(p)], S


def rolling_compare(Y, step=10, H=10):
    """Conectivitatea totala: OLS VAR(2) pe 250 de zile (indicele capitolului) fata de elastic net si OLS pe 150 de zile."""
    Y = pd.DataFrame(Y)
    rows = {}
    for end in range(250, len(Y) + 1, step):
        d = Y.index[end - 1]
        w250 = Y.iloc[end - 250:end].values
        w150 = Y.iloc[end - 150:end].values
        A, S = var_fit(w250, 2)
        c250 = spillover_table(gfevd(A, S, H), list(Y.columns))['total']
        A, S = var_fit(w150, 1)
        c150 = spillover_table(gfevd(A, S, H), list(Y.columns))['total']
        A, S = var_enet_window(w150, 1)
        ce = spillover_table(gfevd(A, S, H), list(Y.columns))['total']
        A1, S1 = var_fit(w250, 1)
        c250p1 = spillover_table(gfevd(A1, S1, H), list(Y.columns))['total']
        rows[d] = {'ols250_p2': c250, 'ols250_p1': c250p1, 'ols150': c150, 'enet150': ce,
                   'bic': var_bic(w250, 3)}
    return pd.DataFrame(rows).T


def fig_spill_enet(rc):
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    ax.plot(rc.index, rc['ols250_p2'], color=Purple, lw=1.0, label='OLS VAR(2), 250-day windows (chapter index)')
    ax.plot(rc.index, rc['ols150'], color=Orange, lw=0.8, label='OLS VAR(1), 150-day windows')
    ax.plot(rc.index, rc['enet150'], color=Forest, lw=1.0, label='Elastic-net VAR(1), 150-day windows, 10-fold CV')
    ax.set_ylabel('Total connectedness (%)')
    ax.set_xlim(rc.index[0], rc.index[-1])
    legend_outside_bottom(ax, ncol=1, y=-0.15)
    save_fig('ch18_spill_enet')


def granger_pairs(Y, p=1, extra=0):
    """Test Wald robust (HC0) pentru j -> i: regresia lui y_i pe p (+extra) intarzieri ale lui y_i si y_j;
    extra = 1 da testul cu intarzieri suplimentare (Toda-Yamamoto), valid si pentru serii foarte persistente."""
    Y = pd.DataFrame(Y)
    L = p + extra
    out = []
    for i in Y.columns:
        for j in Y.columns:
            if i == j:
                continue
            d = pd.concat({f'{v}{l}': Y[v].shift(l) for v in (i, j) for l in range(1, L + 1)}, axis=1)
            d['y'] = Y[i]
            d = d.dropna()
            X = sm.add_constant(d.drop(columns='y'))
            f = sm.OLS(d['y'], X).fit(cov_type='HC0')
            names = [f'{j}{l}' for l in range(1, p + 1)]
            w = f.wald_test(' , '.join(f'{nm} = 0' for nm in names), scalar=True)
            out.append((j, i, float(w.pvalue)))
    return out


def granger_fdr():
    res = {}
    for lab, Y, p, ex in [('ret', R[ALL], 1, 0), ('vol', V, 1, 1)]:
        g = granger_pairs(Y, p, ex)
        pv = np.array([x[2] for x in g])
        res[lab] = {'m': len(pv), 'raw': int((pv < 0.05).sum()), 'bonf': int((pv < 0.05 / len(pv)).sum()),
                    'holm': int((holm(pv) < 0.05).sum()), 'bh': int(bh(pv, 0.05).sum()),
                    'expected_false_raw': 0.05 * len(pv)}
    return res

## Run

In [ ]:
p = var_bic(V.values, 5)
tot, net = spill_bootstrap(V.values, p)   # several minutes
st = spillover_table(gfevd(*var_fit(V.values, p), 10), ALL)
lo, hi = fig_spill_ci(st['net'].values, net)
print('total', round(st['total'], 1), np.percentile(tot, [2.5, 97.5]).round(1))
A, S, share = var_aenet(V.values, p)
print('adaptive elastic net', spillover_table(gfevd(A, S, 10), ALL)['total'], share)
rc = rolling_compare(V, step=10)
fig_spill_enet(rc)
print(rc.describe().round(1))
granger_fdr()

![ch18_spill_ci](./ch18_spill_ci.png)

Output: `ch18_spill_ci.pdf`

![ch18_spill_enet](./ch18_spill_enet.png)

Output: `ch18_spill_enet.pdf`